In [1]:
import copy
import math
import random
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# 1. Policy Network (2 hidden layers, ReLU)
class PolicyNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_actions):
        super(PolicyNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, num_actions)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)


# 2. Car Environment
class CarEnv:
    """
    Locked state/action spec:
    - State: [x, y, vx, vy, s1, s2, ..., sk]
    - Actions: 0=turn_left, 1=turn_right, 2=speed_up, 3=no_action
    """

    def __init__(self, num_sensors=5, max_steps=500):
        assert num_sensors in {1, 3, 5, 7, 9}, "num_sensors must be one of {1,3,5,7,9}"
        self.num_sensors = num_sensors
        self.max_steps = max_steps

        # Competition parameters
        self.min_speed = 0.001
        self.max_speed = 0.1
        self.turn_delta = 0.01
        self.turn_speed_factor = 0.9875   # -1.25%
        self.speedup_factor = 1.025       # +2.5%
        self.crash_speed_factor = 0.125   # reduce to 12.5%

        # Sensor fan in front of the car
        self.sensor_fov = math.pi / 2.0
        self.sensor_max_range = math.sqrt(2.0)

        self.reset()

    def reset(self):
        self.x = 0.5
        self.y = 0.5
        self.angle = 0.0
        self.speed = 0.01
        self.step_count = 0
        self.crash_count = 0
        self.total_distance = 0.0
        return self._get_state()

    def _ray_distance_to_walls(self, ray_angle):
        dx = math.cos(ray_angle)
        dy = math.sin(ray_angle)
        eps = 1e-12
        candidates = []

        if abs(dx) > eps:
            t = (0.0 - self.x) / dx
            if t >= 0:
                y_hit = self.y + t * dy
                if 0.0 <= y_hit <= 1.0:
                    candidates.append(t)
            t = (1.0 - self.x) / dx
            if t >= 0:
                y_hit = self.y + t * dy
                if 0.0 <= y_hit <= 1.0:
                    candidates.append(t)

        if abs(dy) > eps:
            t = (0.0 - self.y) / dy
            if t >= 0:
                x_hit = self.x + t * dx
                if 0.0 <= x_hit <= 1.0:
                    candidates.append(t)
            t = (1.0 - self.y) / dy
            if t >= 0:
                x_hit = self.x + t * dx
                if 0.0 <= x_hit <= 1.0:
                    candidates.append(t)

        if not candidates:
            return self.sensor_max_range
        return min(candidates)

    def _get_sensor_readings(self):
        if self.num_sensors == 1:
            angles = [self.angle]
        else:
            left = self.angle - self.sensor_fov / 2.0
            step = self.sensor_fov / (self.num_sensors - 1)
            angles = [left + i * step for i in range(self.num_sensors)]

        readings = []
        for ray_angle in angles:
            d = self._ray_distance_to_walls(ray_angle)
            readings.append(min(d / self.sensor_max_range, 1.0))
        return readings

    def _get_state(self):
        vx = self.speed * math.cos(self.angle)
        vy = self.speed * math.sin(self.angle)
        return np.array([self.x, self.y, vx, vy] + self._get_sensor_readings(), dtype=np.float32)

    def step(self, action):
        self.step_count += 1

        # Action effects
        if action == 0:
            self.angle -= self.turn_delta
            self.speed *= self.turn_speed_factor
        elif action == 1:
            self.angle += self.turn_delta
            self.speed *= self.turn_speed_factor
        elif action == 2:
            self.speed *= self.speedup_factor
        elif action == 3:
            pass
        else:
            raise ValueError("action must be in {0,1,2,3}")

        self.speed = max(self.min_speed, min(self.speed, self.max_speed))

        old_x, old_y = self.x, self.y
        new_x = self.x + self.speed * math.cos(self.angle)
        new_y = self.y + self.speed * math.sin(self.angle)

        crashed = (new_x < 0.0 or new_x > 1.0 or new_y < 0.0 or new_y > 1.0)
        if crashed:
            self.crash_count += 1
            self.speed *= self.crash_speed_factor
            self.speed = max(self.min_speed, min(self.speed, self.max_speed))

        self.x = min(max(new_x, 0.0), 1.0)
        self.y = min(max(new_y, 0.0), 1.0)

        frame_distance = math.sqrt((self.x - old_x) ** 2 + (self.y - old_y) ** 2)
        self.total_distance += frame_distance

        done = self.step_count >= self.max_steps
        info = {
            "frame_distance": frame_distance,
            "crashed": crashed,
            "crash_count": self.crash_count,
            "total_distance": self.total_distance,
            "speed": self.speed,
        }
        return self._get_state(), frame_distance, done, info


def shape_reward(raw_reward: float, info: Dict, next_state: np.ndarray) -> float:
    # Strongly reward clean motion while penalizing crashes and wall hugging.
    min_sensor = float(np.min(next_state[4:]))
    wall_penalty = 0.04 * max(0.0, (0.25 - min_sensor) / 0.25)
    crash_penalty = 0.25 if info["crashed"] else 0.0
    speed_bonus = 0.03 * info["speed"]
    survival_bonus = 0.002
    shaped = (3.0 * raw_reward) + speed_bonus + survival_bonus - wall_penalty - crash_penalty
    return float(shaped)


def generate_trajectory(network, env):
    trajectory = []
    state = env.reset()
    done = False

    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        logits = network(state_tensor)

        action_dist = torch.distributions.Categorical(logits=logits)
        action_tensor = action_dist.sample()
        action = int(action_tensor.item())
        log_prob = action_dist.log_prob(action_tensor)
        entropy = action_dist.entropy()

        next_state, raw_reward, done, info = env.step(action)
        reward = shape_reward(raw_reward, info, next_state)

        trajectory.append({
            "state": state.copy(),
            "action": action,
            "reward": reward,
            "raw_reward": raw_reward,
            "log_prob": log_prob,
            "entropy": entropy,
            "info": info,
        })
        state = next_state

    return trajectory


def compute_discounted_returns(rewards, gamma=0.97):
    """Compute per-time-step returns G_t = r_t + gamma*r_{t+1} + ..."""
    returns = []
    running = 0.0
    for r in reversed(rewards):
        running = r + gamma * running
        returns.append(running)
    returns.reverse()
    return torch.tensor(returns, dtype=torch.float32)


def train_reinforce(
    network,
    optimizer,
    trajectories,
    gamma=0.97,
    normalize_advantages=True,
    entropy_coef=0.01,
    grad_clip=1.0,
):
    """
    Stabilized REINFORCE:
    - discounted returns
    - per-trajectory baseline (advantages)
    - entropy bonus for exploration
    - gradient clipping
    """
    optimizer.zero_grad()

    total_policy_loss = torch.tensor(0.0)
    total_entropy = torch.tensor(0.0)
    total_steps = 0
    all_returns: List[float] = []

    for trajectory in trajectories:
        rewards = [step["reward"] for step in trajectory]
        returns = compute_discounted_returns(rewards, gamma=gamma)
        all_returns.extend(returns.tolist())

        # Baseline subtraction -> advantages
        advantages = returns - returns.mean()
        if normalize_advantages and len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        for step, adv in zip(trajectory, advantages):
            total_policy_loss = total_policy_loss + (-(step["log_prob"] * adv))
            total_entropy = total_entropy + step["entropy"]

        total_steps += len(trajectory)

    if total_steps == 0:
        return {
            "loss": 0.0,
            "policy_loss": 0.0,
            "entropy": 0.0,
            "return_mean": 0.0,
            "return_std": 0.0,
            "grad_norm": 0.0,
        }

    avg_policy_loss = total_policy_loss / total_steps
    avg_entropy = total_entropy / total_steps
    loss = avg_policy_loss - (entropy_coef * avg_entropy)

    loss.backward()
    grad_norm = float(torch.nn.utils.clip_grad_norm_(network.parameters(), grad_clip).item())
    optimizer.step()

    return {
        "loss": float(loss.item()),
        "policy_loss": float(avg_policy_loss.item()),
        "entropy": float(avg_entropy.item()),
        "return_mean": float(np.mean(all_returns)) if all_returns else 0.0,
        "return_std": float(np.std(all_returns)) if all_returns else 0.0,
        "grad_norm": grad_norm,
    }


def evaluate_policy(network, env, episodes=20, deterministic=True):
    """
    Competition metrics + diagnostic metrics.
    """
    distance_scores = []
    crash_scores = []
    final_speed_scores = []

    for _ in range(episodes):
        state = env.reset()
        done = False

        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32)
            with torch.no_grad():
                logits = network(state_tensor)
                if deterministic:
                    action = int(torch.argmax(logits).item())
                else:
                    action = int(torch.distributions.Categorical(logits=logits).sample().item())
            state, _, done, _ = env.step(action)

        distance_scores.append(env.total_distance)
        crash_scores.append(env.crash_count)
        final_speed_scores.append(env.speed)

    return {
        "mode": "det" if deterministic else "stoch",
        "avg_distance": float(np.mean(distance_scores)),
        "avg_crashes": float(np.mean(crash_scores)),
        "crash_free_rate": float(np.mean([c == 0 for c in crash_scores])),
        "avg_final_speed": float(np.mean(final_speed_scores)),
    }


def evaluate_policy_bundle(network, env, episodes=30):
    return {
        "det": evaluate_policy(network, env, episodes=episodes, deterministic=True),
        "stoch": evaluate_policy(network, env, episodes=episodes, deterministic=False),
    }


def score_for_selection(metrics: Dict) -> float:
    # Optimize for higher distance and lower crashes.
    det = metrics["det"]
    return (det["avg_distance"] * 3.0) - (det["avg_crashes"] * 1.0) + (det["crash_free_rate"] * 2.0)


def save_policy(network, path, metadata=None):
    payload = {
        "state_dict": network.state_dict(),
        "metadata": metadata or {},
    }
    torch.save(payload, path)


def load_policy(path, input_size, hidden_size, num_actions, map_location="cpu"):
    payload = torch.load(path, map_location=map_location)
    model = PolicyNetwork(input_size=input_size, hidden_size=hidden_size, num_actions=num_actions)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    return model, payload.get("metadata", {})


def predict_action(network, state, deterministic=True):
    state_tensor = torch.tensor(state, dtype=torch.float32)
    with torch.no_grad():
        logits = network(state_tensor)
        if deterministic:
            return int(torch.argmax(logits).item())
        return int(torch.distributions.Categorical(logits=logits).sample().item())

In [2]:
# 1. Reproducibility
SEED = 42
set_global_seed(SEED)

# 2. Choose sensor count k (must be odd in {1,3,5,7,9})
k = 7
input_size = 4 + k  # [x, y, vx, vy] + k sensors
hidden_size = 64
num_actions = 4

# 3. Create environment and network
# Train on medium horizon for faster iteration, evaluate on same horizon
max_steps = 120
train_env = CarEnv(num_sensors=k, max_steps=max_steps)
eval_env = CarEnv(num_sensors=k, max_steps=max_steps)

net = PolicyNetwork(input_size=input_size, hidden_size=hidden_size, num_actions=num_actions)
optimizer = optim.Adam(net.parameters(), lr=0.003, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)

print("Environment and policy initialized")
print("Seed:", SEED)
print("State size:", input_size)
print("Max steps per episode:", max_steps)
print("Optimizer: Adam(lr=0.003, weight_decay=1e-5)")
print("Action mapping: 0=left, 1=right, 2=speed_up, 3=no_action")

Environment and policy initialized
Seed: 42
State size: 11
Max steps per episode: 120
Optimizer: Adam(lr=0.003, weight_decay=1e-5)
Action mapping: 0=left, 1=right, 2=speed_up, 3=no_action


In [3]:
# --- Real trajectory and metrics sanity check ---
state0 = train_env.reset()
print("Initial state length:", len(state0), "expected:", 4 + k)

state_tensor = torch.tensor(state0, dtype=torch.float32)
with torch.no_grad():
    logits = net(state_tensor)
    probs = torch.softmax(logits, dim=0)

print("Network output shape:", tuple(logits.shape))
print("Action probabilities:", probs.numpy())
print("Probability sum:", float(probs.sum()))

# Run one full trajectory to ensure environment + trajectory builder are connected
traj = generate_trajectory(net, train_env)
traj_reward = sum(step["reward"] for step in traj)
traj_raw_reward = sum(step["raw_reward"] for step in traj)
traj_crashes = traj[-1]["info"]["crash_count"] if len(traj) > 0 else 0
print("Trajectory length:", len(traj))
print("Trajectory shaped reward:", traj_reward)
print("Trajectory raw distance reward:", traj_raw_reward)
print("Trajectory crash count:", traj_crashes)

# Quick deterministic + stochastic evaluation sample
quick_bundle = evaluate_policy_bundle(net, eval_env, episodes=8)
print("Quick deterministic metrics:", quick_bundle["det"])
print("Quick stochastic metrics:", quick_bundle["stoch"])

Initial state length: 11 expected: 11
Network output shape: (4,)
Action probabilities: [0.24461433 0.24357058 0.24439237 0.26742265]
Probability sum: 1.0
Trajectory length: 120
Trajectory shaped reward: -18.13725803897629
Trajectory raw distance reward: 0.5030160881487984
Trajectory crash count: 66
Quick deterministic metrics: {'mode': 'det', 'avg_distance': 0.5, 'avg_crashes': 71.0, 'crash_free_rate': 0.0, 'avg_final_speed': 0.001}
Quick stochastic metrics: {'mode': 'stoch', 'avg_distance': 0.503668010374957, 'avg_crashes': 68.75, 'crash_free_rate': 0.0, 'avg_final_speed': 0.001}


In [4]:
# --- Full training system: curriculum-ish settings, stabilization, checkpointing ---
num_epochs = 80
games_per_epoch = 16
gamma = 0.97
init_entropy_coef = 0.02
min_entropy_coef = 0.002
entropy_decay = 0.97

eval_every = 5
eval_episodes = 24
early_stop_patience = 8

# Baseline evaluation before training
baseline = evaluate_policy_bundle(net, eval_env, episodes=eval_episodes)
baseline_score = score_for_selection(baseline)
print("Before training (det):", baseline["det"])
print("Before training (stoch):", baseline["stoch"])
print("Baseline score:", round(baseline_score, 4))
print()

history = []
best_score = baseline_score
best_epoch = 0
epochs_without_improvement = 0
best_state = copy.deepcopy(net.state_dict())

for epoch in range(1, num_epochs + 1):
    entropy_coef = max(min_entropy_coef, init_entropy_coef * (entropy_decay ** (epoch - 1)))

    batch_trajectories = [generate_trajectory(net, train_env) for _ in range(games_per_epoch)]
    train_stats = train_reinforce(
        net,
        optimizer,
        batch_trajectories,
        gamma=gamma,
        normalize_advantages=True,
        entropy_coef=entropy_coef,
        grad_clip=1.0,
    )
    scheduler.step()

    row = {
        "epoch": epoch,
        "lr": float(optimizer.param_groups[0]["lr"]),
        "entropy_coef": entropy_coef,
        **train_stats,
    }

    # Periodic evaluation and best checkpoint tracking
    if epoch % eval_every == 0:
        eval_metrics = evaluate_policy_bundle(net, eval_env, episodes=eval_episodes)
        score = score_for_selection(eval_metrics)
        row["det_avg_distance"] = eval_metrics["det"]["avg_distance"]
        row["det_avg_crashes"] = eval_metrics["det"]["avg_crashes"]
        row["det_crash_free_rate"] = eval_metrics["det"]["crash_free_rate"]
        row["score"] = score

        improved = score > (best_score + 1e-6)
        if improved:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = copy.deepcopy(net.state_dict())
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:3d}/{num_epochs} | "
            f"loss={train_stats['loss']:.5f} | "
            f"ret_mean={train_stats['return_mean']:.4f} | "
            f"ent={train_stats['entropy']:.4f} | "
            f"grad={train_stats['grad_norm']:.4f} | "
            f"det_dist={eval_metrics['det']['avg_distance']:.4f} | "
            f"det_crash={eval_metrics['det']['avg_crashes']:.4f} | "
            f"score={score:.4f}"
        )

        if epochs_without_improvement >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch} (no improvement for {early_stop_patience} evals).")
            history.append(row)
            break

    history.append(row)

# Restore best model
net.load_state_dict(best_state)

# Final evaluation
after = evaluate_policy_bundle(net, eval_env, episodes=40)
after_score = score_for_selection(after)

print("\nBest epoch:", best_epoch)
print("Best score:", round(best_score, 4))
print("After training (det):", after["det"])
print("After training (stoch):", after["stoch"])
print("After score:", round(after_score, 4))

print("\nDeterministic deltas from baseline:")
print("Avg distance delta:", after["det"]["avg_distance"] - baseline["det"]["avg_distance"])
print("Avg crashes delta:", after["det"]["avg_crashes"] - baseline["det"]["avg_crashes"])
print("Crash-free rate delta:", after["det"]["crash_free_rate"] - baseline["det"]["crash_free_rate"])

# Save best model for submission/inference
model_path = "policy_network_best.pt"
save_policy(
    net,
    model_path,
    metadata={
        "k": k,
        "input_size": input_size,
        "hidden_size": hidden_size,
        "num_actions": num_actions,
        "max_steps": max_steps,
        "seed": SEED,
        "best_epoch": best_epoch,
        "best_score": best_score,
    },
)
print("\nSaved best model to:", model_path)

# Quick load test
loaded_net, loaded_meta = load_policy(model_path, input_size, hidden_size, num_actions)
test_action = predict_action(loaded_net, eval_env.reset(), deterministic=True)
print("Loaded model metadata:", loaded_meta)
print("Sample predicted action from loaded model:", test_action)

Before training (det): {'mode': 'det', 'avg_distance': 0.5, 'avg_crashes': 71.0, 'crash_free_rate': 0.0, 'avg_final_speed': 0.001}
Before training (stoch): {'mode': 'stoch', 'avg_distance': 0.5034067631275269, 'avg_crashes': 70.04166666666667, 'crash_free_rate': 0.0, 'avg_final_speed': 0.001}
Baseline score: -69.5

Epoch   5/80 | loss=-0.02127 | ret_mean=-4.9782 | ent=1.3815 | grad=0.0281 | det_dist=0.5670 | det_crash=56.0000 | score=-54.2989
Epoch  10/80 | loss=-0.01807 | ret_mean=-5.0262 | ent=1.3758 | grad=0.0125 | det_dist=0.5830 | det_crash=21.0000 | score=-19.2511
Epoch  15/80 | loss=-0.01427 | ret_mean=-5.0182 | ent=1.3768 | grad=0.0222 | det_dist=0.5830 | det_crash=21.0000 | score=-19.2511
Epoch  20/80 | loss=-0.01869 | ret_mean=-5.0064 | ent=1.3804 | grad=0.0266 | det_dist=0.5830 | det_crash=21.0000 | score=-19.2511
Epoch  25/80 | loss=-0.01386 | ret_mean=-4.8489 | ent=1.3806 | grad=0.0062 | det_dist=0.5830 | det_crash=21.0000 | score=-19.2511
Epoch  30/80 | loss=-0.00706 | re